# Raw to Silver: Data Preprocessing
This notebook demonstrates how to preprocess transaction data from the raw layer and save a cleaned version to the silver layer. You can adapt this workflow for other raw datasets in your project.

In [52]:


import pandas as pd

RAW_PATH = "../raw-data/interactions_dataset.csv"
SILVER_PATH = "../Silver-data/interactions_clean.csv"

# Load raw interactions data
interactions = pd.read_csv(RAW_PATH)
print("Raw interactions Shape:", interactions.shape)
interactions.head()

Raw interactions Shape: (3150, 6)


,interaction_id,user_id,product_id,interaction_type,interaction_ts,interaction_date
0,1.0,253.0,554.0,view,2024-06-11 11:42:34.120871,2024-01-08
1,2.0,380.0,NaN,view,2024-02-10 13:37:54.088955,2024-06-06
2,3.0,179.0,1299.0,add_to_cart,2024-01-05 06:15:17.568281,2024-04-27
3,4.0,353.0,2020.0,view,2024-08-30 21:28:41.425280,2024-02-17
4,5.0,54.0,878.0,view,2024-03-18 04:13:30.783131,2024-06-09


## Step 1: Remove Duplicates

In [53]:
# Remove duplicate rows
interactions = interactions.drop_duplicates()
print("After removing duplicates:", interactions.shape)

After removing duplicates: (3045, 6)


## Step 2: Handle Missing Values

In [54]:
# Check for missing values
missing_summary = interactions.isnull().sum()
print("Missing values per column:\n", missing_summary)


Missing values per column:
 interaction_id       81
user_id              93
product_id           99
interaction_type     90
interaction_ts      114
interaction_date     96
dtype: int64


In [55]:
interactions['interaction_id'] = range(1, len(interactions) + 1)
#Drop ONLY rows where user_id OR product_id is null
interactions = interactions.dropna(subset=['user_id', 'product_id'])


In [56]:
#filling interaction_type missing values with mode

# Find mode
mode_interaction = interactions['interaction_type'].mode()[0]

# Fill missing values
interactions['interaction_type'] = interactions['interaction_type'].fillna(mode_interaction)


In [57]:
# Filling missing interaction_ts with interaction_date
interactions['interaction_ts'] = interactions['interaction_ts'].fillna(
    pd.to_datetime(interactions['interaction_date'])
)

# If interaction_ts is still missing, fill with current timestamp
interactions['interaction_ts'] = interactions['interaction_ts'].fillna(
    pd.Timestamp.now()
)
# Create interaction_date from interaction_ts
interactions['interaction_date'] = pd.to_datetime(
    interactions['interaction_ts']
).dt.date


In [58]:
interactions.shape

(2856, 6)

## Step 3: Data Type Corrections (if needed)

In [59]:
interactions.head(100)



,interaction_id,user_id,product_id,interaction_type,interaction_ts,interaction_date
0,1,253.0,554.0,view,2024-06-11 11:42:34.120871,2024-06-11
2,3,179.0,1299.0,add_to_cart,2024-01-05 06:15:17.568281,2024-01-05
3,4,353.0,2020.0,view,2024-08-30 21:28:41.425280,2024-08-30
4,5,54.0,878.0,view,2024-03-18 04:13:30.783131,2024-03-18
5,6,200.0,2061.0,wishlist,2024-10-19 18:48:53.290430,2024-10-19
...,...,...,...,...,...,...
102,103,73.0,799.0,view,2024-07-07 02:01:38.687686,2024-07-07
104,105,58.0,2072.0,view,2024-06-26 02:09:47.915234,2024-06-26
105,106,444.0,2451.0,view,2024-08-08 23:43:46.005064,2024-08-08
106,107,255.0,1684.0,wishlist,2024-03-16 03:48:37.308928,2024-03-16


In [60]:
interactions.dtypes

interaction_id        int64
user_id             float64
product_id          float64
interaction_type     object
interaction_ts       object
interaction_date     object
dtype: object

In [61]:
# IDs → int
interactions['user_id'] = interactions['user_id'].astype('int64')
interactions['product_id'] = interactions['product_id'].astype('int64')

# interaction_type → category (optional but best)
interactions['interaction_type'] = interactions['interaction_type'].astype('category')

# timestamps → datetime
interactions['interaction_ts'] = pd.to_datetime(
    interactions['interaction_ts'], errors='coerce'
)

interactions['interaction_date'] = pd.to_datetime(
    interactions['interaction_date'], errors='coerce'
).dt.date


## Step 4: Save Cleaned Data to Silver Layer

In [62]:
# Save the cleaned dataframe to the silver layer
import os
os.makedirs(os.path.dirname(SILVER_PATH), exist_ok=True)
interactions.to_csv(SILVER_PATH, index=False)
print(f"Cleaned interactions data saved to {SILVER_PATH}")

Cleaned interactions data saved to ../Silver-data/interactions_clean.csv


---

Repeat this process for each raw dataset (users, products, shops, etc.) by changing the input/output paths and adapting the cleaning steps as needed.